# 04 · Per-parcel basemap inspector

Enter a **`COD_PREDIO`** (the record id of the polygon dataset,
`data/processed/training_crop_polygon.parquet`) and this notebook reproduces
**Section 8 of notebook 01** for that one parcel: it draws the parcel **and its
neighbours** on **three basemaps** —

1. **Landsat-5/7** (30 m), the median composite for the **year the target crop was
   reported** (the only sensor that covers the ~1998–2007 titling years);
2. **Sentinel-2** (10 m), the earliest low-cloud scene (2015–16) — a *location* check;
3. **Esri World Imagery** (recent, sub-metre) — the sharpest *location* check.

Every polygon is outlined and captioned with **its crop(s)** and **its reported year**,
with a colour legend keyed to the crop set.

**How to use:** set `RECORD_ID` (and optionally `HALF_WIDTH_DEG`) in the config cell,
then *Run All*. Re-run just the last cell with a different id via
`inspect_parcel("<COD_PREDIO>")`.

> Caveats carried from notebook 01 §8: the **year is the titling/registration date**
> (unreliable, batch/stub values), so only the Landsat panel is even *year*-matched;
> Sentinel-2 and Esri are years-to-decades later and speak only to **where** the parcel
> is, not what grew on it. Crops are shown as the **normalised list** from the pipeline,
> including non-crop states (fallow / land-prep) which are kept, not dropped.

In [ ]:
# --- config, imports, data load, Earth Engine init ---
import io, urllib.request, warnings, socket
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.image as mpimg
import contextily as cx
from shapely.geometry import box
warnings.filterwarnings("ignore", category=UserWarning)
socket.setdefaulttimeout(90)          # no tile/GEE fetch can hang the run

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
POLY_PATH = ROOT / "data" / "processed" / "training_crop_polygon.parquet"
parcels = gpd.read_parquet(POLY_PATH)                 # EPSG:4326, one row / COD_PREDIO
parcels["crops"] = parcels["crops"].map(list)         # parquet -> python lists
print(f"loaded {len(parcels):,} parcels from {POLY_PATH.name}  (CRS {parcels.crs.to_epsg()})")

# Earth Engine is needed only for the Landsat & Sentinel-2 panels. If it is not
# authenticated the notebook still runs and shows the Esri panel.
GEE_PROJECT = "peru-crop-classifier"                   # <- your EE-enabled GCP project
try:
    import ee
    ee.Initialize(project=GEE_PROJECT)
    GEE_OK = True
    print("Earth Engine ready; project:", GEE_PROJECT)
except Exception as e:                                 # noqa: BLE001
    GEE_OK = False
    print("Earth Engine unavailable -> Landsat/Sentinel panels will be skipped.\n ", repr(e))

# ============================ THE KNOBS ====================================
RECORD_ID = "7_5959435_52255"     # <- enter any COD_PREDIO from the dataset here
HALF_WIDTH_DEG = 0.0015           # AOI half-size (~265 m); raise to see more neighbours
# ===========================================================================

In [ ]:
# --- basemap fetchers: year-matched Landsat-5/7 and earliest Sentinel-2 (Earth Engine) ---
def _mask_l5(img):
    # Collection-2 QA_PIXEL (same bits L5 & L7): bit 3 = cloud, bit 4 = cloud shadow
    qa = img.select("QA_PIXEL")
    clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return img.updateMask(clear)

def landsat_rgb(bbox, year, dim=700):
    """Cloud-masked Landsat-5/7 median true-colour composite for `year`.
    bbox = [minlon, minlat, maxlon, maxlat]. Returns (rgb_array, extent)."""
    import ee
    region = ee.Geometry.Rectangle(bbox)
    def _prep(coll_id):
        return (ee.ImageCollection(coll_id).filterBounds(region)
                  .filterDate(f"{year}-01-01", f"{year}-12-31").map(_mask_l5))
    # merge L5 + L7 (identical SR_B* naming); L7 fills L5 gaps in some year/locations
    col = _prep("LANDSAT/LT05/C02/T1_L2").merge(_prep("LANDSAT/LE07/C02/T1_L2"))
    rgb = (col.median().select(["SR_B3", "SR_B2", "SR_B1"])      # red/green/blue
              .multiply(0.0000275).add(-0.2))                    # C2 SR scaling
    url = rgb.getThumbURL({"region": region, "dimensions": dim, "format": "png",
                           "min": 0.0, "max": 0.3, "crs": "EPSG:4326"})
    arr = mpimg.imread(io.BytesIO(urllib.request.urlopen(url, timeout=90).read()))
    return arr, [bbox[0], bbox[2], bbox[1], bbox[3]]

def sentinel2_rgb(bbox, dim=700, max_cloud=20, start="2015-06-23", end="2018-12-31"):
    """Earliest low-cloud Sentinel-2 (harmonized TOA) true-colour scene over bbox.
    Returns (rgb_array, extent, date_str)."""
    import ee
    region = ee.Geometry.Rectangle(bbox)
    col = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(region)
             .filterDate(start, end)
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
             .sort("system:time_start"))
    img = col.first()
    date = img.date().format("YYYY-MM-dd").getInfo()
    rgb = img.select(["B4", "B3", "B2"]).divide(10000)          # 10 m R/G/B
    url = rgb.getThumbURL({"region": region, "dimensions": dim, "format": "png",
                           "min": 0.0, "max": 0.3, "crs": "EPSG:4326"})
    arr = mpimg.imread(io.BytesIO(urllib.request.urlopen(url, timeout=90).read()))
    return arr, [bbox[0], bbox[2], bbox[1], bbox[3]], date

In [ ]:
# --- parcel selection, colours, overlay drawing, and the inspect_parcel() driver ---
# A fixed qualitative palette; crop-sets are coloured in frequency order so the most
# common crops keep the boldest colours across every run.
PALETTE = ["#00e5ff", "#ffe600", "#ff2079", "#7CFC00", "#ff8c00", "#c77dff", "#00ff9f",
           "#ff5555", "#4dabf7", "#ffd6a5", "#b5179e", "#f4a261", "#2ec4b6", "#e71d36",
           "#a0e548", "#ff99c8", "#9bf6ff", "#fdffb6", "#bdb2ff", "#ffc6ff"]

def crop_label(crops):
    return " + ".join(crops) if len(crops) else "(none)"

def year_str(y):
    return "?" if pd.isna(y) else str(int(y))

def aoi_bbox(target_geom, half_width):
    """Square AOI centred on the parcel, widened so the parcel always fits with margin."""
    b = target_geom.bounds
    hw = max(half_width, 0.75 * max(b[2] - b[0], b[3] - b[1]))
    cx_, cy_ = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    return [cx_ - hw, cy_ - hw, cx_ + hw, cy_ + hw]

def select_window(record_id, half_width):
    """Return (target_row, bbox, window_gdf) for a COD_PREDIO. window = all parcels
    whose bounding box falls in the AOI (i.e. the target and its neighbours)."""
    if record_id not in set(parcels.COD_PREDIO):
        raise KeyError(f"COD_PREDIO {record_id!r} is not in {POLY_PATH.name}")
    target = parcels.loc[parcels.COD_PREDIO == record_id].iloc[0]
    bbox = aoi_bbox(target.geometry, half_width)
    win = parcels.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]].copy()
    win["crop_label"] = win.crops.map(crop_label)
    win["is_target"] = win.COD_PREDIO == record_id
    return target, bbox, win

def colour_map(win):
    order = win.crop_label.value_counts().index.tolist()
    return {lab: PALETTE[i % len(PALETTE)] for i, lab in enumerate(order)}

def _overlay(ax, win, colours, to_3857, annotate):
    """Draw parcel outlines coloured by crop-set, highlight the target, and caption
    each parcel with its crop(s) + reported year."""
    g = win.to_crs(3857) if to_3857 else win
    for lab, part in g.groupby("crop_label"):
        part.boundary.plot(ax=ax, color=colours[lab], linewidth=1.8, zorder=2)
    tgt = g[g.is_target]                                     # target on top, white halo
    tgt.boundary.plot(ax=ax, color="white", linewidth=3.6, zorder=3)
    tgt.boundary.plot(ax=ax, color=colours[tgt.iloc[0].crop_label], linewidth=1.6, zorder=4)
    if annotate:
        for _, r in g.iterrows():
            c = r.geometry.centroid
            ax.annotate(f"{r.crop_label}\n{year_str(r.year)}", (c.x, c.y), color="white",
                        fontsize=7.2 if r.is_target else 6.0,
                        fontweight="bold" if r.is_target else "normal",
                        ha="center", va="center", zorder=5,
                        path_effects=[pe.withStroke(linewidth=1.6, foreground="black")])

def _legend_handles(win, colours, record_id):
    counts = win.crop_label.value_counts()
    handles = [mpatches.Patch(color=colours[l], label=f"{l} ({n})") for l, n in counts.items()]
    handles.append(mpatches.Patch(facecolor="none", edgecolor="white", linewidth=2.6,
                                  label=f"★ target {record_id}"))
    return handles

def inspect_parcel(record_id=None, half_width=None, annotate=True):
    """Section-8 view for one parcel on three basemaps. Returns the per-parcel summary."""
    record_id = record_id or RECORD_ID
    half_width = HALF_WIDTH_DEG if half_width is None else half_width
    target, bbox, win = select_window(record_id, half_width)
    colours = colour_map(win)

    print(f"target {record_id}: crops={list(target.crops)}  year={year_str(target.year)}"
          f"  area_ha={target.area_ha:.3f}  codigo_sset={list(target.codigo_sset)}")
    print(f"AOI half-width {half_width:.4f}°  |  {len(win)} parcels in window, "
          f"{win.crop_label.nunique()} distinct crop-sets")

    fig, axes = plt.subplots(1, 3, figsize=(19, 6.4), dpi=150)

    # panel 1 — year-matched Landsat-5/7
    ax = axes[0]
    if GEE_OK:
        yr = int(target.year) if not pd.isna(target.year) else 2001
        try:
            img, ext = landsat_rgb(bbox, yr)
            ax.imshow(img, extent=ext, origin="upper", zorder=0)
            title = f"Landsat-5/7 {year_str(target.year)}  (30 m, year-matched)"
        except Exception as e:                              # noqa: BLE001
            ax.text(0.5, 0.5, f"Landsat fetch failed:\n{e}", transform=ax.transAxes, ha="center")
            title = "Landsat-5/7 (fetch failed)"
        _overlay(ax, win, colours, to_3857=False, annotate=annotate)
        ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
    else:
        title = "Landsat-5/7 (Earth Engine unavailable)"
    ax.set_title(title, fontsize=10)

    # panel 2 — earliest Sentinel-2 (location check)
    ax = axes[1]
    if GEE_OK:
        try:
            img, ext, s2date = sentinel2_rgb(bbox)
            ax.imshow(img, extent=ext, origin="upper", zorder=0)
            title = f"Sentinel-2 {s2date}  (10 m, location check)"
        except Exception as e:                              # noqa: BLE001
            ax.text(0.5, 0.5, f"Sentinel-2 fetch failed:\n{e}", transform=ax.transAxes, ha="center")
            title = "Sentinel-2 (fetch failed)"
        _overlay(ax, win, colours, to_3857=False, annotate=annotate)
        ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
    else:
        title = "Sentinel-2 (Earth Engine unavailable)"
    ax.set_title(title, fontsize=10)

    # panel 3 — recent sub-metre Esri (location check)
    ax = axes[2]
    b3857 = gpd.GeoSeries([box(*bbox)], crs=4326).to_crs(3857).total_bounds
    _overlay(ax, win, colours, to_3857=True, annotate=annotate)
    ax.set_xlim(b3857[0], b3857[2]); ax.set_ylim(b3857[1], b3857[3])
    try:
        cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, zoom=17, zorder=0)
    except Exception as e:                                  # noqa: BLE001
        ax.text(0.5, 0.5, f"Esri basemap failed:\n{e}", transform=ax.transAxes, ha="center")
    ax.set_title("Esri World Imagery (recent, sub-metre, location check)", fontsize=10)

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    fig.legend(handles=_legend_handles(win, colours, record_id), loc="center left",
               bbox_to_anchor=(0.865, 0.5), fontsize=8, framealpha=0.9,
               title="crops (parcel count)")
    fig.suptitle(f"Parcel {record_id} + neighbours — crop & reported year on three basemaps",
                 fontsize=13, y=0.99)
    fig.tight_layout(rect=[0, 0, 0.86, 0.96]); plt.show()

    summary = (win[["COD_PREDIO", "is_target", "crop_label", "year", "area_ha", "n_records"]]
               .sort_values("is_target", ascending=False).reset_index(drop=True))
    return summary

In [ ]:
# --- run for RECORD_ID; the returned table is the crops + year of every parcel shown ---
summary = inspect_parcel(RECORD_ID, HALF_WIDTH_DEG)
summary

### Inspect another parcel

Call the driver directly with any `COD_PREDIO` — no need to edit the config cell:

```python
inspect_parcel("7_5259400_66194")                 # a mango/lime/guabo cluster
inspect_parcel("7_4959460_50191", half_width=0.0018)   # tighter zoom
```

Reading the three panels together (notebook 01 §8): **Landsat** is coarse but the only
*year-matched* view; **Sentinel-2** and **Esri** are sharper but years-to-decades later,
so they confirm *where* the parcel is, not *what* grew on it. The target parcel is the one
with the bold white outline and the ★ legend entry.

In [ ]:
# sort parcels by year for easier inspection of the dataset
parcels = parcels.sort_values("year")
parcels.tail(10)

In [ ]:
newest_record_id = parcels.loc[parcels.year.idxmax(), "COD_PREDIO"]
print(f"newest parcel in dataset: {newest_record_id}  year={parcels.year.max()}")

In [ ]:

summary = inspect_parcel('7_5759450_030671', HALF_WIDTH_DEG)
summary

In [ ]:
summary = inspect_parcel('7_5859440_079425', half_width=0.0008)
summary

In [ ]:
summary = inspect_parcel('7_4959460_50191', HALF_WIDTH_DEG)
summary